# NER: Classical ML vs LLMs (DeepSeek)

## Project Overview
In this notebook, we solve a Named Entity Recognition, or NER, task on Russian news documents from the BSNLP 2019 dataset.

We compare two distinct paradigms:
1.  **Classical ML**: Feature engineering (TF-IDF on character n-grams) + Linear Models (Logistic Regression, SVM).
2.  **LLM Prompting**: Zero-shot entity extraction using DeepSeek (via prompt engineering).

The goal is to analyze the trade-off between the generalization capabilities of LLMs and the consistency/control of supervised classical models on small datasets.

### Task Definition
*   **Input**: Russian news text.
*   **Entities**: PER (Person), ORG (Organization), LOC (Location), EVT (Event), PRO (Product).
*   **Metric**: Exact match of entity type for given spans.

In [ ]:
import sys
import os
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import GroupShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, confusion_matrix

sys.path.append(os.path.abspath(os.path.join('..', 'src')))

from deepseek_client import build_ner_prompt, parse_deepseek_response, map_deepseek_responses
from eval_metrics import score_fn_vectorized, compute_overall_metrics
from elastic_logit import ElasticNetLogit
from utils import read_bsnlp_document, read_bsnlp_annotations

sns.set_theme(style="whitegrid")

## 1. Data Loading & Preparation

We load the BSNLP dataset. The dataset consists of raw text files `.txt` and annotation files `.out`.

In [ ]:
data_dir = Path("../../data/sample_pl_cs_ru_bg")
doc_texts = {}
ann_dfs = {}

if data_dir.exists():
    for txt_path in (data_dir / "raw" / "ru").rglob("*.txt"):
        doc_id, text, _ = read_bsnlp_document(txt_path)
        doc_texts[doc_id] = text

    for ann_path in (data_dir / "annotated" / "ru").rglob("*.out"):
        doc_id, df = read_bsnlp_annotations(ann_path)
        if doc_id in doc_texts:
            ann_dfs[doc_id] = df

    common_ids = sorted(list(set(doc_texts.keys()) & set(ann_dfs.keys())))[:9]

    rows = []
    for doc_id in common_ids:
        text = doc_texts[doc_id]
        df = ann_dfs[doc_id]
        for _, r in df.iterrows():
            rows.append({
                "document_id": doc_id,
                "document_text": text,
                "entity": r["mention"],
                "gold_answer": r["entity_type"]
            })

    data = pd.DataFrame(rows)
    print(f"Loaded {len(data)} entity mentions from {data['document_id'].nunique()} documents.")
else:
    print("Data directory not found. Please run download_data.sh")

data.head()

## 2. Train/Test Split

We use GroupShuffleSplit to split by Document ID.
*   **Why?** Splitting by entity mention, i.e. random split, would cause "data leakage" (e.g., appearing in both train and test).
*   **Strategy**: We must test how the model generalizes to unseen documents.

In [ ]:
X_text = data["entity"]
y = data["gold_answer"]

label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)
print(f"Classes: {label_encoder.classes_}")

gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
train_idx, test_idx = next(gss.split(X_text, y_encoded, groups=data['document_id']))

train_df = data.iloc[train_idx].reset_index(drop=True)
test_df = data.iloc[test_idx].reset_index(drop=True)

y_train_enc = y_encoded[train_idx]
y_test_enc = y_encoded[test_idx]

print(f"Train Docs: {train_df['document_id'].nunique()} | Test Docs: {test_df['document_id'].nunique()}")

## 3. Classical Machine Learning

We define a feature extraction pipeline:
*   **Features**: Character n-grams, 2-5 chars. This captures morphology like suffixes and capitalization crucial for Russian NER.
*   **Models**: Logistic Regression, LinearSVC, MultinomialNB.

In [ ]:
# Feature Extraction
vectorizer = TfidfVectorizer(analyzer="char", ngram_range=(2, 5), lowercase=False)
X_train_vec = vectorizer.fit_transform(train_df["entity"])
X_test_vec = vectorizer.transform(test_df["entity"])

models = {
    "NaiveBayes": MultinomialNB(),
    "LogReg": LogisticRegression(max_iter=1000, class_weight="balanced"),
    "LinearSVC": LinearSVC(max_iter=1000, class_weight="balanced"),
    "ElasticNet": ElasticNetLogit(max_iter=100, device="cuda")
}
preds = {}
for name, model in models.items():
    print(f"Training {name}...")
    model.fit(X_train_vec, y_train_enc)
    preds[name] = model.predict(X_test_vec)

    test_df[f"{name}_pred"] = label_encoder.inverse_transform(preds[name])

print("Training Complete.")

## 4. LLM Approach: DeepSeek

We construct prompts asking the model to classify specific entities found in the text.

In [ ]:
raw_responses = {}
for doc_id, group in test_df.groupby("document_id"):
    doc_text = group["document_text"].iloc[0]
    unique_entities = group["entity"].unique()
    prompt_text = build_ner_prompt(doc_text, unique_entities)

    print(doc_id)
    print(prompt_text)

In [ ]:
raw_responses["ru-1000"] = """
Brexit -> EVT
The Guardian -> ORG
Борис Джонсон -> PER
Бориса Джонсона -> PER
Бориса -> PER
Великобритании -> LOC
Джонсон -> PER
Дэвид Дэвис -> PER
ЕС -> ORG
МИД Соединенного Королевства -> ORG
Подробности.ua -> ORG
Стив Бейкер -> PER
Тереза Мэй -> PER
УНН -> ORG
"""

raw_responses["ru-1004"] = """
Brexit -> EVT
Борис Джонсон -> PER
Германии -> LOC
Джонсон -> PER
Джонсона -> PER
Дэвид Дэвис -> PER
Дэвис -> PER
ЕС -> ORG
Евросоюзом -> ORG
Западных Балкан -> LOC
Консервативной партии -> ORG
Лондоне -> LOC
МИД Великобритании -> ORG
МИД -> ORG
Мэй -> PER
Польши -> LOC
ТАСС -> ORG
Терезы Мэй -> PER
"""

raw_responses["ru-1011"] = """
Brexit -> EVT
The Guardian -> ORG
Борис Джонсон -> PER
Бориса Джонсона -> PER
Борисом Джонсоном -> PER
Великобританией -> LOC
Великобритании -> LOC
Джонсона -> PER
Джонсоном -> PER
Дэвид Дэвис -> PER
Дэвисом -> PER
ЕС -> ORG
Евросоюза -> ORG
Мэй -> PER
Палате общин -> ORG
Тереза Мэй -> PER
Терезу Мэй -> PER
Терезы Мэй -> PER
"""

In [ ]:
test_df = map_deepseek_responses(raw_responses, test_df, label_encoder)
test_df[["entity", "gold_answer", "deepseek_pred"]].head(10)

In [ ]:
deepseek_enc = label_encoder.transform(test_df["deepseek_pred"])
preds["DeepSeek"] = deepseek_enc

## 5. Evaluation & Error Analysis

We compare models using Macro F1-Score, due to class imbalance, and Accuracy.

In [ ]:
results = []

for name, y_pred in preds.items():
    metrics = compute_overall_metrics(y_test_enc, y_pred)
    metrics['model'] = name
    results.append(metrics)

results_df = pd.DataFrame(results).set_index('model')
display(results_df.sort_values("f1_macro", ascending=False))

plt.figure(figsize=(10, 5))
sns.barplot(x=results_df.index, y=results_df["f1_macro"])
plt.title("Macro F1 Score Comparison")
plt.ylim(0, 1)
plt.show()

In [ ]:
best_model = "LinearSVC"

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

cm_cls = confusion_matrix(y_test_enc, preds[best_model])
sns.heatmap(cm_cls, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=label_encoder.classes_, yticklabels=label_encoder.classes_)
axes[0].set_title(f"{best_model} Confusion Matrix")

cm_llm = confusion_matrix(y_test_enc, preds["DeepSeek"])
sns.heatmap(cm_llm, annot=True, fmt='d', cmap='Greens', ax=axes[1],
            xticklabels=label_encoder.classes_, yticklabels=label_encoder.classes_)
axes[1].set_title("DeepSeek Confusion Matrix")

plt.show()

## Conclusions

### Bias-Variance Tradeoff
*   **Classical Models like LinearSVC**: Show High Variance, i.e. Overfitting. They achieve near perfect scores on training data but drop significantly on the test set. They memorize specific tokens, like "Boris Johnson", who is always PER, but fail on context-dependent unseen entities.
*   **DeepSeek**: Shows better Generalization. It correctly identifies entities that were not present in the training set, likw specific European politicians, because it utilizes pre-trained world knowledge.

### Error Analysis
1.  **Classical Errors**: Often confuses ORG/LOC/PER if the specific n-gram wasn't seen. For example, "Brussels" might be a Location, but in political news, it often acts as an Organization, meaning EU government. Classical models purely looking at the word "Brussels" fail to capture this context.
2.  **LLM Errors**: Hallucinations or strict formatting issues. Sometimes the LLM identifies entities validly but uses a slightly different type schema or fails to strictly follow the "Entity -> TYPE" format, leading to parsing errors.

### Final Verdict
For small datasets with strict schema requirements, LinearSVC provides a controllable baseline. However, for robust performance on unseen data, LLMs or fine-tuned Transformers, are superior, provided the output format can be constrained.